In [19]:
%pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-bcebtqpx
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-bcebtqpx
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [20]:
%%cuda_group_save -g "pimontecarlo" -n "pimontecarlo.h"
#pragma once

double estimatePIMonteCarloCPU(long long TP, double &elapsedSeconds);
double estimatePIMonteCarloGPU(long long TP, double &elapsedSeconds);

In [21]:
%%cuda_group_save -g "pimontecarlo" -n "pimontecarlo_cpu.cu"
#include <random>
#include <chrono>
#include "pimontecarlo.h"

namespace
{
    double estimatePIMonteCarloImpl(long long TP)
    {
        long long PIC = 0;

        // std::random_device rd;
        std::mt19937 gen(12345);

        std::uniform_real_distribution<double> dist(-1.0, 1.0);

        for (long long i = 0; i < TP; ++i)
        {
            double x = dist(gen);
            double y = dist(gen);

            if ((x * x) + (y * y) <= 1.0)
            {
                PIC++;
            }
        }

        double pi_estimate = 4.0 * static_cast<double>(PIC) / static_cast<double>(TP);

        return pi_estimate;
    }
}

double estimatePIMonteCarloCPU(long long TP, double &elapsedSeconds)
{
    auto start = std::chrono::steady_clock::now();
    double pi_estimate = estimatePIMonteCarloImpl(TP);
    auto end = std::chrono::steady_clock::now();

    std::chrono::duration<double> elapsed = end - start;
    elapsedSeconds = elapsed.count();

    return pi_estimate;
}


In [22]:
%%cuda_group_save -g "pimontecarlo" -n "pimontecarlo_gpu.cu"
#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>
#include <curand_kernel.h>
#include "pimontecarlo.h"

namespace
{
    __global__
    void setupKernel(curandState *states)
    {
        long long tid = static_cast<long long>(blockIdx.x) * blockDim.x + threadIdx.x;
        curand_init(12345, tid, 0, &states[tid]);
    }

    __global__ 
    void estimatePIKernel(long long TP, unsigned long long *PIC, curandState *states)
    {
        long long tid = static_cast<long long>(blockIdx.x) * blockDim.x + threadIdx.x;
        long long stride = static_cast<long long>(gridDim.x) * blockDim.x;

        curandState gen = states[tid];

        unsigned long long hits = 0;

        for (long long i = tid; i < TP; i += stride)
        {
            
            float x = curand_uniform(&gen) * 2.0f - 1.0f;
            float y = curand_uniform(&gen) * 2.0f - 1.0f;

            if ((x * x) + (y * y) <= 1.0f)
            {
                hits++;
            }
        }

        // states[tid] = gen;

        __shared__ unsigned long long sData[256];
        sData[threadIdx.x] = hits;
        __syncthreads();

        for (int s = blockDim.x / 2; s > 0; s >>= 1)
        {
            if (threadIdx.x < s)
                sData[threadIdx.x] += sData[threadIdx.x + s];
            __syncthreads();
        }

        if (threadIdx.x == 0)
            atomicAdd(PIC, sData[0]);
    }

    double estimatePIMonteCarloImpl(long long TP, double &elapsedSeconds)
    {
        const int threadsPerBlock = 256;
        const int numBlocks = static_cast<int>(std::min<long long>((TP + threadsPerBlock - 1) / threadsPerBlock, 1024));

        static curandState *d_states = nullptr;
        if (d_states == nullptr)
        {
            cudaMalloc(&d_states, 1024 * threadsPerBlock * sizeof(curandState));
            setupKernel<<<1024, threadsPerBlock>>>(d_states);
            cudaDeviceSynchronize();
        }
        
        unsigned long long *d_PIC;
        unsigned long long PIC = 0;

        cudaMalloc(&d_PIC, sizeof(unsigned long long));
        cudaMemset(d_PIC, 0, sizeof(unsigned long long));

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);
        estimatePIKernel<<<numBlocks, threadsPerBlock>>>(TP, d_PIC, d_states);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float milliseconds = 0.0f;
        cudaEventElapsedTime(&milliseconds, start, stop);
        elapsedSeconds = milliseconds / 1000.0;

        cudaMemcpy(&PIC, d_PIC, sizeof(unsigned long long), cudaMemcpyDeviceToHost);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
        cudaFree(d_PIC);

        double pi_estimate = 4.0 * static_cast<double>(PIC) / static_cast<double>(TP);

        return pi_estimate;
    }
}

double estimatePIMonteCarloGPU(long long TP, double &elapsedSeconds)
{
    // Warm up the GPU to avoid measuring initialization overhead in the first run

    static bool warmedUp = false;
    if (!warmedUp)
    {
        double ignored = 0.0;
        estimatePIMonteCarloImpl(1, ignored);
        warmedUp = true;
    }

    return estimatePIMonteCarloImpl(TP, elapsedSeconds);
}


In [23]:
%%cuda_group_save -g "pimontecarlo" -n "main.cu"
#include <cmath>
#include <cstdlib>
#include <format>
#include <iostream>
#include <numbers>
#include "pimontecarlo.h"

int main()
{
    long long TP = 100;

    std::cout << std::format(
        "{:<14} {:<14} {:<14} {:<14} {:<14} {:<14} {:<14} {:<10}\n",
        "N",
        "Pi (CPU)",
        "Pi (GPU)",
        "Error (CPU)",
        "Error (GPU)",
        "CPU Time (s)",
        "GPU Time (s)",
        "Speedup");

    std::cout << std::format(
        "{:-<14} {:-<14} {:-<14} {:-<14} {:-<14} {:-<14} {:-<14} {:-<10}\n",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "");

    for (int i = 0; i < 8; ++i)
    {
        double cpu_time = 0.0;
        double gpu_time = 0.0;

        double pi_cpu = estimatePIMonteCarloCPU(TP, cpu_time);
        double pi_gpu = estimatePIMonteCarloGPU(TP, gpu_time);

        double error_cpu = std::abs(pi_cpu - std::numbers::pi);
        double error_gpu = std::abs(pi_gpu - std::numbers::pi);
        double speedup = cpu_time / gpu_time;

        std::cout << std::format(
            "{:<14} {:<14.10f} {:<14.10f} {:<14.10f} {:<14.10f} {:<14.6f} {:<14.6f} {:<10.1f}\n",
            TP,
            pi_cpu,
            pi_gpu,
            error_cpu,
            error_gpu,
            cpu_time,
            gpu_time,
            speedup);

        TP *= 10;
    }

    return EXIT_SUCCESS;
}


In [24]:
%cuda_group_run --group "pimontecarlo" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

N              Pi (CPU)       Pi (GPU)       Error (CPU)    Error (GPU)    CPU Time (s)   GPU Time (s)   Speedup   
-------------- -------------- -------------- -------------- -------------- -------------- -------------- ----------
100            2.6800000000   3.3200000000   0.4615926536   0.1784073464   0.000003       0.000012       0.3       
1000           3.0960000000   3.1240000000   0.0455926536   0.0175926536   0.000017       0.000012       1.4       
10000          3.1344000000   3.1056000000   0.0071926536   0.0359926536   0.000153       0.000012       12.5      
100000         3.1431600000   3.1378000000   0.0015673464   0.0037926536   0.001558       0.000027       57.3      
1000000        3.1412800000   3.1405120000   0.0003126536   0.0010806536   0.014521       0.000058       250.4     
10000000       3.1415580000   3.1417292000   0.0000346536   0.0001365464   0.147805       0.000100       1481.4    
100000000      3.1417225200   3.1418661600   0.0001298664   0.0002735064

In [30]:
!find /tmp/tmpgmub28wy -type f

/tmp/tmpgmub28wy/pimontecarlo/pimontecarlo_gpu.cu
/tmp/tmpgmub28wy/pimontecarlo/pimontecarlo_cpu.cu
/tmp/tmpgmub28wy/pimontecarlo/main.cu
/tmp/tmpgmub28wy/pimontecarlo/cuda_exec.out
/tmp/tmpgmub28wy/pimontecarlo/pimontecarlo.h


In [32]:
!nvcc -O3 -lineinfo -std=c++20 -arch=sm_75 -I/tmp/tmpgmub28wy /tmp/tmpgmub28wy/pimontecarlo/*.cu -o pi

In [34]:
!./pi

N              Pi (CPU)       Pi (GPU)       Error (CPU)    Error (GPU)    CPU Time (s)   GPU Time (s)   Speedup   
-------------- -------------- -------------- -------------- -------------- -------------- -------------- ----------
100            2.6800000000   3.3200000000   0.4615926536   0.1784073464   0.000005       0.000013       0.4       
1000           3.0960000000   3.1240000000   0.0455926536   0.0175926536   0.000031       0.000015       2.0       
10000          3.1344000000   3.1056000000   0.0071926536   0.0359926536   0.000158       0.000017       9.5       
100000         3.1431600000   3.1378000000   0.0015673464   0.0037926536   0.001462       0.000032       45.5      
1000000        3.1412800000   3.1405120000   0.0003126536   0.0010806536   0.016677       0.000062       267.5     
10000000       3.1415580000   3.1417292000   0.0000346536   0.0001365464   0.145919       0.000113       1290.7    
100000000      3.1417225200   3.1418661600   0.0001298664   0.0002735064

In [36]:
!ncu --set full -k regex:estimatePIKernel --launch-skip 8 --launch-count 1 -o pi_kernel ./pi

N              Pi (CPU)       Pi (GPU)       Error (CPU)    Error (GPU)    CPU Time (s)   GPU Time (s)   Speedup   
-------------- -------------- -------------- -------------- -------------- -------------- -------------- ----------
==PROF== Connected to process 8889 (/content/pi)
100            2.6800000000   3.3200000000   0.4615926536   0.1784073464   0.000005       0.003592       0.0       
1000           3.0960000000   3.1240000000   0.0455926536   0.0175926536   0.000026       0.000450       0.1       
10000          3.1344000000   3.1056000000   0.0071926536   0.0359926536   0.000277       0.002534       0.1       
100000         3.1431600000   3.1378000000   0.0015673464   0.0037926536   0.004786       0.003631       1.3       
1000000        3.1412800000   3.1405120000   0.0003126536   0.0010806536   0.035338       0.000569       62.1      
10000000       3.1415580000   3.1417292000   0.0000346536   0.0001365464   0.276300       0.000678       407.6     
100000000      3.141722

In [37]:
!ncu --import pi_kernel.ncu-rep --page details

[8889] pi@127.0.0.1
  <unnamed>::estimatePIKernel(long long, unsigned long long *, curandStateXORWOW *) (1024, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    6,352,078
    Memory Throughput                 %         0.46
    DRAM Throughput                   %         0.46
    Duration                         ms        10.86
    L1/TEX Cache Throughput           %         0.33
    L2 Cache Throughput               %         0.14
    SM Active Cycles              cycle 6,245,683.80
    Compute (SM) Throughput           %        87.74
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater t